In [1]:
import os
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import models
from pydantic import SecretStr
from langchain_openrouter import ChatOpenRouter
from rag_utils import print_wrapped

In [2]:
# 1. Beispieldokumente mit Metadaten definieren
docs = [
    Document(
        page_content="Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt.",
        metadata={"source": "handbuch_v1.pdf", "kategorie": "Wartung", "jahr": 2025}
    ),
    Document(
        page_content="Bei einem Temperaturanstieg im Hauptkessel muss das System sofort gekühlt werden.",
        metadata={"source": "sicherheit.pdf", "kategorie": "Sicherheit", "jahr": 2026}
    ),
    Document(
        page_content="Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck-Dichtungsring.",
        metadata={"source": "katalog.pdf", "kategorie": "Ersatzteile", "jahr": 2026}
    ),
    Document(
        page_content="Regelmäßige Inspektionen verhindern Ausfälle der Hydraulikpumpe.",
        metadata={"source": "wartungsplan.pdf", "kategorie": "Wartung", "jahr": 2024}
    )
]

In [3]:
# 2. Dense- & Sparse-Embedding-Modelle initialisieren
dense_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [4]:
# 3. VARIANTE A: Rein Semantischer Vector Store (DENSE)
semantic_vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=dense_embeddings,
    location=":memory:",
    collection_name="rag_semantisch",
    retrieval_mode=RetrievalMode.DENSE,
)
semantic_retriever = semantic_vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 2}
)

In [5]:
# 4. VARIANTE B: Nativer Hybrider Vector Store (DENSE + SPARSE)
hybrid_vectorstore = QdrantVectorStore.from_documents(
    documents=docs,
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
    location=":memory:",
    collection_name="rag_hybrid",
    retrieval_mode=RetrievalMode.HYBRID,
)
hybrid_retriever = hybrid_vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 2}
)

In [6]:
# 5. Metadaten-Filterung für Qdrant definieren (Wartung ab Jahr 2025)
qdrant_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="metadata.kategorie",
            match=models.MatchValue(value="Wartung"),
        ),
        models.FieldCondition(
            key="metadata.jahr",
            range=models.Range(gte=2025),
        ),
    ]
)
filtered_hybrid_retriever = hybrid_vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 2, "filter": qdrant_filter}
)

In [7]:
# 6. Test der Suchmodi (Semantisch, Hybrid & Gefiltert)
query = "Was bedeutet ERR-4029 oder Probleme am Ventil?"

print("=== 1. Semantische Suche ===")
for d in semantic_retriever.invoke(query):
    print(f"- {d.page_content} | Meta: {d.metadata}")

print("\n=== 2. Hybride Suche (Dense + BM25) ===")
for d in hybrid_retriever.invoke(query):
    print(f"- {d.page_content} | Meta: {d.metadata}")
    
print("\n=== 3. Hybride Suche + Metadaten-Filter (Wartung >= 2025) ===")
for d in filtered_hybrid_retriever.invoke("Inspektion und Wartung"):
    print(f"- {d.page_content} | Meta: {d.metadata}")

=== 1. Semantische Suche ===
- Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt. | Meta: {'source': 'handbuch_v1.pdf', 'kategorie': 'Wartung', 'jahr': 2025, '_id': 'be3fad0e-e8d4-43ce-a2e8-eb550ecdcce6', '_collection_name': 'rag_semantisch'}
- Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck-Dichtungsring. | Meta: {'source': 'katalog.pdf', 'kategorie': 'Ersatzteile', 'jahr': 2026, '_id': 'b7394df0-3ac6-4bd3-a744-bdd45ba5ccde', '_collection_name': 'rag_semantisch'}

=== 2. Hybride Suche (Dense + BM25) ===
- Der Fehlercode ERR-4029 tritt auf, wenn das Druckventil überhitzt. | Meta: {'source': 'handbuch_v1.pdf', 'kategorie': 'Wartung', 'jahr': 2025, '_id': '3d6c8f9b-e664-4bbd-99c0-ff9a747c2afa', '_collection_name': 'rag_hybrid'}
- Das Ersatzteil mit der Artikelnummer ART-88912 ist ein Hochdruck-Dichtungsring. | Meta: {'source': 'katalog.pdf', 'kategorie': 'Ersatzteile', 'jahr': 2026, '_id': 'd47ebf43-1472-48a8-9143-994b87f4cfe8', '_collection_name': 'rag_h

In [8]:
# 7. Einbindung in eine komplette RAG-Chain (LCEL)
def format_docs(retrieved_docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', 'Unbekannt')}] {d.page_content}"
        for d in retrieved_docs
    )

In [9]:
prompt = ChatPromptTemplate.from_template("""
    Beantworte die Frage ausschließlich auf Basis des folgenden Kontexts.
    Nenne am Ende kurz die verwendete Quelle.

    Kontext:
    {context}
    Frage: {question}
""")

In [10]:
api_key = SecretStr(os.getenv("OPENROUTER_API_KEY", ""))
if not api_key.get_secret_value():
    raise RuntimeError("OPENROUTER_API_KEY is not set")

In [11]:
qwen = "qwen/qwen3-30b-a3b-instruct-2507"
llm = ChatOpenRouter(
    api_key=api_key,
    model=qwen,
    temperature=0,
)

In [12]:
# Beliebiges LangChain-ChatModel einbinden (z. B. ChatOllama oder ChatOpenAI):
def build_rag_chain(retriever, llm):
    return (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )
# Beispielaufruf:
rag_chain = build_rag_chain(filtered_hybrid_retriever, llm)
print_wrapped(rag_chain.invoke("Was tun bei ERR-4029?"))

Bei ERR-4029, dem Fehlercode für ein überhitztes Druckventil, sollte das Gerät
abgeschaltet und abkühlen lassen. Danach sollte überprüft werden, ob das
Druckventil ordnungsgemäß funktioniert und keine Verstopfungen oder andere
Störungen vorliegen. Falls der Fehler weiterhin auftritt, ist ein Fachmann
hinzuzuziehen.

Quelle: handbuch_v1.pdf
